In [16]:
# Data Science Lab Assignment 7: Data Transformation & Error Correction
# Dataset: Raw Titanic train.csv

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

print("Original shape:", df.shape)
print("Missing %%:\n", (df.isnull().mean() * 100).round(2))

# 1. ERROR CORRECTION: Duplicates
print("\nDuplicates:", df.duplicated().sum())
df_clean = df.drop_duplicates().reset_index(drop=True)

# 2. MISSING IMPUTATION (no inplace warnings)
df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
print("Missing after:", df_clean.isnull().sum().sum())

# 3. OUTLIER CAP IQR (Fare)
Q1, Q3 = df_clean['Fare'].quantile([0.25,0.75])
IQR = Q3 - Q1
bounds = [Q1-1.5*IQR, Q3+1.5*IQR]
outliers = ((df_clean['Fare'] < bounds[0]) | (df_clean['Fare'] > bounds[1])).sum()
df_clean['Fare_cap'] = df_clean['Fare'].clip(bounds[0], bounds[1])
print("Fare outliers:", outliers)

# 4. TRANSFORMATIONS
df_clean['Fare_log'] = np.log1p(df_clean['Fare'])
df_clean['Age_bin'] = pd.cut(df_clean['Age'], bins=5, labels=['Ch','Te','Yo','Ad','Sr'])

# 5. SCALING
mm = MinMaxScaler(); df_clean['Age_mm'] = mm.fit_transform(df_clean[['Age']]).flatten()
z = StandardScaler(); df_clean['Fare_z'] = z.fit_transform(df_clean[['Fare_cap']]).flatten()

# 6. ENCODING
le = LabelEncoder(); df_clean['Sex_le'] = le.fit_transform(df_clean['Sex'])

print("\nSkew Fare: %.2f -> log: %.2f" % (df_clean['Fare'].skew(), df_clean['Fare_log'].skew()))
print("Age range: %.0f -> MM[0,1]" % (df_clean['Age'].max()-df_clean['Age'].min()))
print("Fare_z: μ=%.3f σ=%.3f" % (df_clean['Fare_z'].mean(), df_clean['Fare_z'].std()))
print("\nTransformed sample:")
print(df_clean[['Age','Age_mm','Fare','Fare_log','Sex','Sex_le','Age_bin']].head())

print("\nSummary:\n- Dupes removed\n- Impute Age(grp med), Emb(Mode), Fare(med)\n- Cap %d Fare outliers\n- Log Fare, bin Age\n- Scale MM(Age)/Z(Fare)\n- Label Sex"%outliers)


Original shape: (891, 12)
Missing %%:
 PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

Duplicates: 0
Missing after: 687
Fare outliers: 116

Skew Fare: 4.79 -> log: 0.39
Age range: 80 -> MM[0,1]
Fare_z: μ=0.000 σ=1.001

Transformed sample:
    Age    Age_mm     Fare  Fare_log     Sex  Sex_le Age_bin
0  22.0  0.271174   7.2500  2.110213    male       1      Te
1  38.0  0.472229  71.2833  4.280593  female       0      Yo
2  26.0  0.321438   7.9250  2.188856  female       0      Te
3  35.0  0.434531  53.1000  3.990834  female       0      Yo
4  35.0  0.434531   8.0500  2.202765    male       1      Yo

Summary:
- Dupes removed
- Impute Age(grp med), Emb(Mode), Fare(med)
- Cap 116 Fare outliers
- Log Fare, bin Age
- Scale MM(Age)/Z(Fare)
- Label Sex
